In [ ]:
#Load packages

import requests
from bs4 import BeautifulSoup
import pandas as pd
import openpyxl
from datetime import datetime
import time
import os


In [ ]:
os.chdir("C:/Users/DELL/Desktop/No Regret/replication codes")

In [ ]:
# Define headers
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 '
                  'Safari/537.36',
    'Cookie': 'csrftoken=so14fKmaX2QkPlDl7ktQtEj5VO9J4Bl1bEd7KTTVfzVDhdZRQATfFiTemSiNJjPx; '
              '__ssid=875d55c025f771618c00e71e9e7ec6d; sessionid=30wuvr8fys45iaxdlpbmwauc93zfcwg3; '
              'ajs_anonymous_id=ffd295434428faa44a800120765b4a8c'
              '-81049a3d20b260cad2ffcc4b3d7b373f7de4dfa16fe73fc779b31e5d83e22b94; '
              '_gac_UA-501243-21=1.1687971478.Cj0KCQjwtO-kBhDIARIsAL6LorefMsoMmap0yGCCakeHfm6onlYBCz8-aQAfSqUajv'
              '-X9lBi_ZUntvUaAnKVEALw_wcB; _gid=GA1.2.368544338.1689002158; _clck=177mhfi|2|fd7|0|1255; '
              '__cf_bm=Z1.2Q3pgLkpSEzRRKuJp5ubbeR06wuZtbo3yZ3tYx_s-1689094753-0-ATzatJZ'
              '+TBwv6HCTYfvepFnZKR2q4ougqhial87RMTjYJFGms3TPZLdxW3EgByLwyQ==; '
              '_ga_MV8J4ZYG1Z=GS1.1.1689094752.26.1.1689095829.0.0.0; _ga=GA1.2.913824145.1686333247; '
              '_clsk=194juxb|1689095829854|19|1|t.clarity.ms/collect'
}

In [ ]:
def scrape_device(device):
    url = 'https://swappa.com/listings/' + str(device) + '?carrier=unlocked&color=&storage=&modeln=&condition=&sort='
    current_url = url
    device_id = str(device)
    df = pd.DataFrame()
    series = 0
    while True:
        # Send HTTP request and retrieve the page content
        response = requests.get(url=current_url, headers=headers)
        if response.status_code == 200:
            # Extract and process the data from the page
            content = response.text
            soup = BeautifulSoup(content, 'html.parser')
            whole_table = soup.find('table', attrs={'id': 'listings_table'})
            # Append the table string to a DataFrame
            new = pd.read_html(str(whole_table))[0]
            df = pd.concat([df, new], ignore_index=True)
            series += 1
            print('scraped completed for device id: ' + f'{device_id}' + '-page-' + f'{series}')

            # Check if there is a next page link
            next_link = soup.find("a", attrs={'class': 'page-link', 'aria-label': 'Next Page', 'title': 'Next Page'})
            if next_link is None:
                # No more pages available, exit the loop
                break

            # Get the URL of the next page and continue scraping
            current_url = url + next_link["href"]
        else:
            print("Error accessing page")
            break 
    df['device_id'] = device_id
    return df

In [ ]:
wb = openpyxl.load_workbook('Data/Raw_Data_Clean/DEVICES.xlsx')
sheet = wb['Sheet1']
devices = sheet['A']
device_list = [device.value for device in devices]  # Create a list of devices

# Scrape the devices
res = pd.DataFrame()
for device in device_list:
    res = pd.concat([res, scrape_device(device)], ignore_index=True) # Scrape the device and concatenate the data to the existing dataframe

# Save the data to a CSV file
res['date'] = datetime.now().strftime("%Y-%m-%d")
filename_date = 'Swappa_Scrape_' + datetime.now().strftime("%m%d") + '_sale.xlsx'
res.to_excel("Data/Swappa_Daily_Data"+filename_date, index=False)


In [ ]:
#scrape the previously unsold codes
wb = pd.read_excel("Data/Raw_Data_clean/Swappa_0711_1223_raw.xlsx") 
wb = wb[['Code']]
wb['Real_Sold'] = ''
wb['Created'] = ''
wb['Updated'] = ''

def scrape_sold(url):
    response = requests.get(url=url, headers=headers)
    sold_text = 'Not_Sold'
    if response.status_code == 200:
        content = response.text
        soup = BeautifulSoup(content, 'html.parser')

        div_element = soup.find('div', class_='btn btn-primary disabled')
        created_text = 'Created date not found'
        updated_text = 'Updated date not found'


        span_elements = soup.find_all('span', class_='m-1')
        for span_element in span_elements:
            i_element = span_element.find('i', class_='far fa-calendar-day')
            if i_element:
                created_text = span_element.get_text(strip=True).replace('Created ', '')

            i_element = span_element.find('i', class_='far fa-calendar-arrow-up')
            if i_element:
                updated_text = span_element.get_text(strip=True).replace('Updated ', '')


        if div_element:
            sold_text = div_element.get_text(strip=True)
            
        return sold_text, created_text, updated_text
    else:
        print(f'Error fetching page: {url}')
        return "Error fetching page", "", ""

#for every code in wb column 'Code', run the scrape_sold function, the url will be 'https://swappa.com/listing/view/' + str(code) and then save the sold_text and updated_text to the corresponding row's column 'Real_Sold', 'Created', 'Updated'
for index, row in wb.iterrows():
    url = 'https://swappa.com/listing/view/' + str(row['Code'])
    sold_text, created_text, updated_text = scrape_sold(url)
    wb.at[index, 'Real_Sold'] = sold_text
    wb.at[index, 'Created'] = created_text
    wb.at[index, 'Updated'] = updated_text
    if index % 10 == 0 and index != 0:
        print(f'Finished {index} of {len(wb)}')

    if index % 1000 == 0 and index != 0:
        print("Pausing for 1 minute...")
        time.sleep(60)  # Pause for 60 seconds (1 minute)

wb.to_excel("Data/Raw_Data_Clean/selling_status.xlsx", index=False) 